## はじめに

このノートブックでは、まず Cortex Code（CoCo）に質問して GlacierStyle ECサイトのデータを知り、続けて音声・PDF を取り込んで **Bronze 層**のテーブルを作成します。音声や PDF のような非構造化データも、ステージに置けば SQL でテーブルにできることを体験します。

> **メダリオンアーキテクチャ**: データを加工の段階ごとに3つの層に分けて管理します。
> - **Bronze 層**: 取り込んだままの生データ（`RAW_` で始まるテーブル）
> - **Silver 層**: 整えて分析に使える形にしたデータ（`DIM_`・`FACT_`、Part 2 の `CALL_ANALYSIS`・`DOC_ATTRIBUTES`）
> - **Gold 層**: 検索や AI アシスタントにそのまま使う形にしたデータ（Part 2 の `KNOWLEDGE_CHUNKS`）

**前提条件:**
- `setup_20261216.sql` が実行済みであること（データベース・スキーマ・ステージ・GitHub 連携の設定、データファイルのコピー、構造化データ（CSV）のロードと、ディメンション／ファクトテーブルへのコメント付与が完了）

**主な処理内容:**
1. Cortex Code（CoCo）に質問してデータを知る
2. 音声ファイルの文字起こし（AI_TRANSCRIBE）
3. PDF ドキュメントの解析（AI_PARSE_DOCUMENT）

**このノートブックで使う・作るテーブル:**

*Silver 層: ディメンションテーブル（マスタデータ）※ setup でロード済み*
- `DIM_CUSTOMERS`: 顧客マスタ（300件）
- `DIM_PRODUCTS`: 商品マスタ（55件）

*Silver 層: ファクトテーブル（トランザクションデータ）※ setup でロード済み*
- `FACT_ORDERS`: EC取引データ（2,913件。入荷ロット番号付き）

*Bronze 層: 非構造化データテーブル*
- `RAW_CALL_TRANSCRIPTS`: 文字起こし済みのコールセンター通話（88件）※ setup でロード済み
- `RAW_VOICE_LOGS`: 通話音声の文字起こし結果（10件）
- `RAW_DOCUMENTS`: PDF ドキュメントのページ単位のテキスト（11ファイル・99ページ）

> **料金について**: AI 関数は処理量に応じてクレジットを消費します。実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。このノートブックの AI 処理は、音声10件と PDF 99ページです。

> **途中で遅れた場合**: `checkpoint_20261216.sql` の「Part 1」を実行すると、このノートブックの完了時点のテーブルを復元できます。

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するロール、ウェアハウス、スキーマを設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 1. Cortex Code（CoCo）に質問してデータを知る

`setup_20261216.sql` で、商品・顧客・注文の CSV と、文字起こし済みの通話がすでにテーブルに入っています。まずは SQL を書かずに、**Cortex Code（CoCo）** に日本語で聞いて、どんなデータがあるのかをつかみましょう。

画面右側の CoCo のチャット欄に、下の質問をコピーして送ってみてください。CoCo は実際にテーブルやステージを調べてから答えます。

### 1-1. どんなデータがあるか聞いてみよう

```
GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA にはどんなデータがありますか？テーブルごとに件数と中身を教えてください
```

```
DIM_CUSTOMERS・DIM_PRODUCTS・FACT_ORDERS・RAW_CALL_TRANSCRIPTS はどの列でつながっていますか？
```

```
ステージ DATA_STAGE にはどんなファイルがありますか？種類ごとに教えてください
```

> **ポイント**: setup で、ディメンションテーブルとファクトテーブルの列に日本語の説明（コメント）を付けてあります。CoCo はこの説明を読んで、`order_status` の `returned` が「返品」であることなどを理解します。

### 1-2. このデータで何ができるか聞いてみよう

```
このデータを使って、EC サイトの売上や顧客対応についてどんな分析ができますか？アイデアを5つ挙げてください
```

```
2026年の月別売上を販売チャネル別に集計してください
```

```
返品率が高い商品を上位5件教えてください
```

> **ヒント**: CoCo が作った SQL はチャット欄に表示されます。気になった SQL は、このノートブックに新しいセルを追加して貼り付けて実行することもできます。

ここで気づいたこと（たとえば、返品が多い商品）は、Part 2 で通話の内容と組み合わせて原因を掘り下げます。そのために、次の章から音声と PDF もテーブルにしていきます。


In [ ]:
-- ============================================================================
-- setup で付与した列のコメントを確認
-- ============================================================================
-- CoCo はこの説明を手がかりにデータの意味を理解します
SELECT table_name, column_name, data_type, comment
FROM INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'EC_ANALYTICS_SCHEMA'
  AND table_name IN ('DIM_PRODUCTS', 'DIM_CUSTOMERS', 'FACT_ORDERS')
ORDER BY table_name, ordinal_position;

## 2. 音声ファイルの文字起こし

コールセンターの通話音声（MP3、10件）を `AI_TRANSCRIBE` で文字起こしします。ファイルは `TO_FILE('@ステージ', 'パス')` で指定します。

この10件は文字起こしを体験するためのサンプルで、注文や商品のデータとはつながっていません。結果は `RAW_VOICE_LOGS` に保存し、Part 2 で文字起こし済みの88件（`RAW_CALL_TRANSCRIPTS`）とあわせて分析します。

In [ ]:
-- ============================================================================
-- AI_TRANSCRIBE: 1件だけ文字起こしを試してみよう
-- ============================================================================
-- 結果は JSON で返り、text に文字起こし、audio_duration に音声の長さ（秒）が入ります
SELECT AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', 'voice_logs/CALL_001.mp3')) AS transcribe_result;

In [ ]:
-- ============================================================================
-- 音声10件を文字起こしし、RAW_VOICE_LOGS に保存
-- ============================================================================
-- DIRECTORY(@DATA_STAGE) でステージ上のファイル一覧を取得し、MP3 だけを処理します
CREATE OR REPLACE TABLE RAW_VOICE_LOGS AS
WITH transcribed AS (
    SELECT
        SPLIT_PART(relative_path, '/', -1) AS audio_file,
        AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', relative_path)) AS result
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE 'voice_logs/%.mp3'
)
SELECT
    'VOICE-' || REPLACE(audio_file, '.mp3', '') AS call_id,
    audio_file,
    result:audio_duration::NUMBER(10, 1) AS audio_duration_sec,
    result:text::VARCHAR AS transcript_text
FROM transcribed;

-- 文字起こし結果を確認
SELECT call_id, audio_duration_sec, LEFT(transcript_text, 80) AS transcript_preview
FROM RAW_VOICE_LOGS
ORDER BY call_id;

## 3. PDF ドキュメントの解析

FAQ、運用マニュアル、品質報告書、通達、店舗報告などの PDF（11ファイル）を `AI_PARSE_DOCUMENT` で解析します。

- `'mode': 'LAYOUT'`: 見出しや表の構造を保ったまま、Markdown 形式のテキストにします
- `'page_split': true`: ページごとに結果を分けます。後で「どのページに書いてあるか」を残せます

In [ ]:
-- ============================================================================
-- AI_PARSE_DOCUMENT: 品質報告書（1ページ）の解析を試してみよう
-- ============================================================================
-- pages の各要素の content に、ページの本文が Markdown で入ります
SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@DATA_STAGE', 'documents/quality_report_202611.pdf'),
    {'mode': 'LAYOUT', 'page_split': true}
) AS parse_result;

In [ ]:
-- ============================================================================
-- PDF 11ファイルをページ単位でテーブルにする
-- ============================================================================
CREATE OR REPLACE TABLE RAW_DOCUMENTS AS
WITH parsed AS (
    SELECT
        SPLIT_PART(relative_path, '/', -1) AS file_name,
        AI_PARSE_DOCUMENT(
            TO_FILE('@DATA_STAGE', relative_path),
            {'mode': 'LAYOUT', 'page_split': true}
        ) AS doc
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE '%.pdf'
)
-- pages の配列を FLATTEN で1ページ1行に展開
SELECT
    p.file_name,
    pg.index + 1 AS page_no,
    pg.value:content::VARCHAR AS page_text
FROM parsed p,
LATERAL FLATTEN(input => p.doc:pages) pg;

-- ファイルごとのページ数と、1ページ目の冒頭を確認
SELECT
    file_name,
    COUNT(*) AS page_count,
    LEFT(MAX(IFF(page_no = 1, page_text, NULL)), 60) AS first_page_preview
FROM RAW_DOCUMENTS
GROUP BY file_name
ORDER BY file_name;

## 4. データ確認

作成した Bronze 層と Silver 層のテーブルの件数を確認します。

In [ ]:
-- ============================================================================
-- 各テーブルのレコード数を確認
-- ============================================================================
SELECT 'DIM_PRODUCTS' AS table_name, COUNT(*) AS record_count, '商品マスタ（55件）' AS description FROM DIM_PRODUCTS
UNION ALL
SELECT 'DIM_CUSTOMERS', COUNT(*), '顧客マスタ（300件）' FROM DIM_CUSTOMERS
UNION ALL
SELECT 'FACT_ORDERS', COUNT(*), '注文データ（2,913件）' FROM FACT_ORDERS
UNION ALL
SELECT 'RAW_CALL_TRANSCRIPTS', COUNT(*), '文字起こし済みの通話（88件）' FROM RAW_CALL_TRANSCRIPTS
UNION ALL
SELECT 'RAW_VOICE_LOGS', COUNT(*), '通話音声の文字起こし（10件）' FROM RAW_VOICE_LOGS
UNION ALL
SELECT 'RAW_DOCUMENTS', COUNT(*), 'PDF のページ（99ページ）' FROM RAW_DOCUMENTS;

## まとめ

このノートブックでは、CoCo に質問してデータの全体像をつかみ、音声と PDF の非構造化データを取り込んで Bronze 層のテーブルを作成しました。

### テーブル一覧

**Silver 層: ディメンションテーブル（マスタデータ）※ setup で作成**
- `DIM_CUSTOMERS`: 顧客マスタ
- `DIM_PRODUCTS`: 商品マスタ

**Silver 層: ファクトテーブル（トランザクションデータ）※ setup で作成**
- `FACT_ORDERS`: EC取引データ

**Bronze 層: 非構造化データテーブル**
- `RAW_CALL_TRANSCRIPTS`: 文字起こし済みの通話（setup で作成）
- `RAW_VOICE_LOGS`: 通話音声（文字起こし済み）
- `RAW_DOCUMENTS`: PDF ドキュメント（ページ単位で解析済み）

### 使用した Cortex AI 関数

- `AI_TRANSCRIBE`: 音声ファイルの文字起こし
- `AI_PARSE_DOCUMENT`: PDF ドキュメントの解析

### 次のステップ

- **Part 2**: Bronze 層のテキストから、AI 関数で問い合わせ種別・感情・解約リスクなどを取り出し、Silver 層と Gold 層のテーブルを作成します